In [13]:
import sys
sys.path.append("../")
import neo4j
import pandas as pd
import pandas as pd
from utils_embeddings import load_embedding_model_std
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.indexes import upsert_vectors
from neo4j_graphrag.types import EntityType
from qdrant_client import QdrantClient
from neo4j_graphrag.retrievers import QdrantNeo4jRetriever
import json
from qdrant_client.models import Filter, FieldCondition, GeoRadius, GeoPoint

In [2]:
model = load_embedding_model_std()

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2


In [3]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [4]:
DIMENSION=384

In [5]:
testEmb = model.encode("This is a test sentence.").tolist()

In [6]:
len(testEmb)

384

In [7]:
COLLECTION_NAME = "simplified_test"

In [9]:
# 1. Verbindungen herstellen
client = QdrantClient("localhost", port=6333)

# 2. Collection erstellen
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "size": model.get_sentence_embedding_dimension(),
        "distance": "Cosine"
    },
)
print(f"2. Qdrant Collection erstellt")

C:\Users\paul-\AppData\Local\Temp\ipykernel_32448\3927554524.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


2. Qdrant Collection erstellt


In [17]:
# 4. Testdaten erstellen
pois = [
    {
        "id": 1,
        "name": "Hamburger Michel",
        "description": "Die St. Michaelis Kirche ist ein bekanntes Wahrzeichen von Hamburg",
        "tags": ["kirche", "sehenswürdigkeit", "hamburg"],
        "coordinates": {"lat": 53.5486, "lon": 9.9793}
    },
    {
        "id": 2,
        "name": "Elbphilharmonie",
        "description": "Konzerthaus und Wahrzeichen der Hamburger Hafencity",
        "tags": ["musik", "konzert", "sehenswürdigkeit", "hamburg"],
        "coordinates": {"lat": 53.5413, "lon": 9.9841}
    },
    {
        "id": 3, 
        "name": "Landungsbrücken",
        "description": "Historische Anlegestellen am Hamburger Hafen",
        "tags": ["hafen", "schiffe", "hamburg"],
        "coordinates": {"lat": 53.5462, "lon": 9.9669}
    },
    {
        "id": 4,
        "name": "St. Petri Kirche",
        "description": "Eine der fünf Hauptkirchen in Hamburg",
        "tags": ["kirche", "hamburg", "altstadt"],
        "coordinates": {"lat": 53.5497, "lon": 9.9975}
    },
    {
        "id": 5,
        "name": "Fischrestaurant Hafenkante",
        "description": "Beliebtes Fischrestaurant mit Blick auf den Hafen",
        "tags": ["restaurant", "fisch", "hamburg", "hafen"],
        "coordinates": {"lat": 53.5450, "lon": 9.9700}
    }
]


# 7. Embeddings erstellen und in Qdrant speichern
for poi in pois:
    # Text für Embedding kombinieren
    combined_text = f"""name: {poi['name']}
Beschreibung: {poi['description']}
Schlagwörter: {', '.join(poi['tags'])}"""
    
    # Embedding erzeugen
    vector = model.encode(combined_text).tolist()
    
    # ID als Integer für Qdrant
    poi_id_int = poi["id"]
    
    # In Qdrant speichern
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=[{
            "id": poi_id_int,  # Integer ID für Qdrant
            "vector": vector,
            "payload": {
                "neo4j_id": str(poi_id_int),  # Als String im payload speichern
                "name": poi["name"],
                "location": {"lon": poi["coordinates"]["lon"], "lat": poi["coordinates"]["lat"]},
            }
        }]
    )

print(f"7. {len(pois)} Embeddings in Qdrant gespeichert")

# 8. Debug: Prüfen der gespeicherten Punkte in Qdrant
collection_info = client.get_collection(collection_name=COLLECTION_NAME)
print(f"\nQdrant Collection '{COLLECTION_NAME}' enthält {collection_info.points_count} Vektoren")




7. 5 Embeddings in Qdrant gespeichert

Qdrant Collection 'simplified_test' enthält 5 Vektoren


In [18]:
search_texts = [
    "Wo kann ich Kirchen in Hamburg besichtigen?",
]
    

# Jede Suchanfrage testen
for search_text in search_texts:
    print(f"\nSuche nach: '{search_text}'")
    
    # Embedding erzeugen
    query_embedding = model.encode(search_text).tolist()
    
    # Suche durchführen
    try:
        filter_query = Filter(
            must=[
                FieldCondition(
                    key="location",
                    geo_radius=GeoRadius(
                        center=GeoPoint(
                            lon=float(9.905292),  # Ensure values are float
                            lat=float(53.544315)
                        ),
                        radius=float(80000)  # Radius in meters, ensure float
                    )
                )
            ]
        )
        # Direkter Check in Qdrant
        qdrant_results = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_embedding,
            limit=2,
            query_filter=filter_query
        )
        
        if qdrant_results:
            print("\nDirekte Qdrant-Ergebnisse:")
            for i, res in enumerate(qdrant_results, 1):
                print(f"Qdrant-Ergebnis {i}:")
                print(f"  ID: {res.id} (Typ: {type(res.id).__name__})")
                print(f"  Payload: {res.payload}")
                print(f"  Score: {res.score:.4f}")
            
    
    except Exception as e:
        print(f"Fehler bei der Suche: {str(e)}")


Suche nach: 'Wo kann ich Kirchen in Hamburg besichtigen?'

Direkte Qdrant-Ergebnisse:
Qdrant-Ergebnis 1:
  ID: 1 (Typ: int)
  Payload: {'neo4j_id': '1', 'name': 'Hamburger Michel', 'location': {'lon': 9.9793, 'lat': 53.5486}}
  Score: 0.7371
Qdrant-Ergebnis 2:
  ID: 4 (Typ: int)
  Payload: {'neo4j_id': '4', 'name': 'St. Petri Kirche', 'location': {'lon': 9.9975, 'lat': 53.5497}}
  Score: 0.7158


C:\Users\paul-\AppData\Local\Temp\ipykernel_32448\2277751411.py:30: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_results = client.search(
